# Garmin Connect Data Retrieval Tool
 
This notebook retrieves data from Garmin Connect and stores it in Delta tables.
 
**Prerequisites:**
- Install required libraries in cluster or use %pip install
- Set up secrets in Databricks Secret Scope for credentials
- Configure Unity Catalog or DBFS for data storage

In [0]:
#%pip install requests python-dateutil


In [0]:
import requests
import json
import pandas as pd
from datetime import datetime, timedelta
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import *
from pyspark.sql.functions import *
import logging
from typing import Dict, List, Optional, Any

# Initialize Spark session (automatically available in Databricks)
spark = SparkSession.builder.getOrCreate()

In [0]:
class GarminConnectAPI:
    """
    Garmin Connect API client optimized for Databricks deployment
    """
    
    def __init__(self, secret_scope: str = "garmin"):
        self.session = requests.Session()
        self.base_url = "https://connect.garmin.com"
        self.modern_url = f"{self.base_url}/modern"
        self.secret_scope = secret_scope
        self.is_authenticated = False
        
        # Set up logging
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)
        
    def authenticate(self) -> bool:
        """
        Authenticate using credentials from Databricks secrets
        """
        try:
            # Get credentials from Databricks secrets
            username = dbutils.secrets.get(scope=self.secret_scope, key="username")
            password = dbutils.secrets.get(scope=self.secret_scope, key="password")
            
            self.logger.info("Attempting to authenticate with Garmin Connect...")
            
            # Get the login form
            login_url = f"{self.base_url}/signin"
            response = self.session.get(login_url)
            
            # Login data
            login_data = {
                'username': username,
                'password': password,
                'embed': 'false'
            }
            
            # Headers to mimic a browser
            headers = {
                'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
                'Content-Type': 'application/x-www-form-urlencoded'
            }
            
            # Submit login
            login_response = self.session.post(
                f"{self.base_url}/signin",
                data=login_data,
                headers=headers,
                allow_redirects=True
            )
            
            # Check if login was successful
            if "dashboard" in login_response.url or login_response.status_code == 200:
                self.is_authenticated = True
                self.logger.info("Successfully authenticated with Garmin Connect")
                return True
            else:
                self.logger.error("Authentication failed")
                return False
                
        except Exception as e:
            self.logger.error(f"Authentication error: {str(e)}")
            return False
    
    def get_activities(self, start_date: str = None, limit: int = 100) -> List[Dict]:
        """
        Retrieve activities from Garmin Connect
        
        Args:
            start_date: Start date in YYYY-MM-DD format
            limit: Maximum number of activities to retrieve
            
        Returns:
            List of activity dictionaries
        """
        if not self.is_authenticated:
            raise Exception("Must authenticate first")
        
        if start_date:
            start_timestamp = int(datetime.strptime(start_date, "%Y-%m-%d").timestamp() * 1000)
        else:
            # Default to last 30 days
            start_timestamp = int((datetime.now() - timedelta(days=30)).timestamp() * 1000)
        
        url = f"{self.modern_url}/proxy/activitylist-service/activities/search/activities"
        params = {
            'start': 0,
            'limit': limit,
            'startDate': start_timestamp
        }
        
        try:
            response = self.session.get(url, params=params)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error retrieving activities: {str(e)}")
            return []
    
    def get_activity_details(self, activity_id: str) -> Dict:
        """
        Get detailed information for a specific activity
        """
        if not self.is_authenticated:
            raise Exception("Must authenticate first")
        
        url = f"{self.modern_url}/proxy/activity-service/activity/{activity_id}"
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error retrieving activity details for {activity_id}: {str(e)}")
            return {}
    
    def get_daily_summary(self, date: str) -> Dict:
        """
        Get daily summary data (steps, calories, etc.)
        
        Args:
            date: Date in YYYY-MM-DD format
        """
        if not self.is_authenticated:
            raise Exception("Must authenticate first")
        
        url = f"{self.modern_url}/proxy/usersummary-service/usersummary/daily/{date}"
        
        try:
            response = self.session.get(url)
            response.raise_for_status()
            return response.json()
        except requests.exceptions.RequestException as e:
            self.logger.error(f"Error retrieving daily summary for {date}: {str(e)}")
            return {}

# COMMAND ----------

# MAGIC %md
# MAGIC ## Table Creation Functions

# COMMAND ----------

def get_activities_schema() -> StructType:
    return StructType([
        StructField('activity_id', LongType(), True),
        StructField('activity_name', StringType(), True),
        StructField('activity_type', StringType(), True),
        StructField('start_time', TimestampType(), True),
        StructField('duration', LongType(), True),
        StructField('distance', DoubleType(), True),
        StructField('calories', LongType(), True),
        StructField('avg_heart_rate', LongType(), True),
        StructField('max_heart_rate', LongType(), True),
        StructField('avg_speed', DoubleType(), True),
        StructField('max_speed', DoubleType(), True),
        StructField('elevation_gain', DoubleType(), True),
        StructField('elevation_loss', DoubleType(), True),
        StructField('retrieved_at', TimestampType(), True)
    ])

def get_daily_summaries_schema() -> StructType:
    return StructType([
        StructField('date', StringType(), True),
        StructField('steps', LongType(), True),
        StructField('calories_burned', LongType(), True),
        StructField('distance_meters', DoubleType(), True),
        StructField('active_minutes', LongType(), True),
        StructField('floors_climbed', LongType(), True),
        StructField('sleep_duration_seconds', LongType(), True),
        StructField('resting_heart_rate', LongType(), True),
        StructField('max_heart_rate', LongType(), True),
        StructField('stress_score', LongType(), True),
        StructField('body_battery', LongType(), True),
        StructField('retrieved_at', TimestampType(), True)
    ])
    
def get_activity_details_schema() -> StructType:
    return StructType([
        StructField('activity_id', LongType(), True),
        StructField('detailed_metrics', StringType(), True),
        StructField('gps_data', StringType(), True),
        StructField('heart_rate_zones', StringType(), True),
        StructField('splits', StringType(), True),
        StructField('retrieved_at', TimestampType(), True)
    ])


def create_garmin_database_and_tables():
    """
    Create Garmin database and tables with explicit schemas.
    """
    try:
        # Create database first
        spark.sql("CREATE DATABASE IF NOT EXISTS garmin")
        print("✅ Created/verified garmin database")
        
        # We don't need to pre-create the tables with DDL if we use `saveAsTable` with the right options.
        # This simplifies the logic. We will let the first write operation create the table.
        return True
    except Exception as e:
        print(f"❌ Error creating database: {str(e)}")
        return False

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Processing Functions

# COMMAND ----------

def process_activities_to_dataframe(activities_data: List[Dict]) -> pd.DataFrame:
    """
    Convert activities data to a pandas DataFrame with proper data types
    """
    if not activities_data:
        return pd.DataFrame()
    
    # Extract relevant fields
    processed_activities = []
    for activity in activities_data:
        processed_activity = {
            'activity_id': activity.get('activityId'),
            'activity_name': activity.get('activityName'),
            'activity_type': activity.get('activityType', {}).get('typeKey'),
            'start_time': pd.to_datetime(activity.get('startTimeLocal')),
            'duration': activity.get('duration'),  # in seconds
            'distance': activity.get('distance'),  # in meters
            'calories': activity.get('calories'),
            'avg_heart_rate': activity.get('averageHR'),
            'max_heart_rate': activity.get('maxHR'),
            'avg_speed': activity.get('averageSpeed'),
            'max_speed': activity.get('maxSpeed'),
            'elevation_gain': activity.get('elevationGain'),
            'elevation_loss': activity.get('elevationLoss'),
            'retrieved_at': datetime.now()
        }
        processed_activities.append(processed_activity)
    
    # Create pandas DataFrame
    df = pd.DataFrame(processed_activities)
    return df

def process_daily_summary_to_dataframe(daily_summaries: List[Dict]) -> pd.DataFrame:
    """
    Convert daily summary data to a pandas DataFrame with proper data types
    Handles the flexible structure of Garmin daily summaries
    """
    if not daily_summaries:
        return pd.DataFrame()
    
    processed_summaries = []
    for summary in daily_summaries:
        # Create a base record with the date
        processed_summary = {
            'date': summary.get('date'),
            'retrieved_at': summary.get('retrieved_at', datetime.now())
        }
        
        # Extract common metrics with safe gets
        processed_summary.update({
            'steps': summary.get('totalSteps'),
            'calories_burned': summary.get('totalKilocalories'), 
            'distance_meters': summary.get('totalDistanceMeters'),
            'active_minutes': summary.get('activeMinutes'),
            'floors_climbed': summary.get('floorsClimbed'),
            'sleep_duration_seconds': summary.get('sleepingSeconds'),
            'resting_heart_rate': summary.get('restingHeartRate'),
            'max_heart_rate': summary.get('maxHeartRate'),
            'stress_score': summary.get('maxStressScore'),
            'body_battery': summary.get('bodyBatteryChargedUp')
        })
        
        processed_summaries.append(processed_summary)
    
    return pd.DataFrame(processed_summaries)

def save_to_delta_table(df: pd.DataFrame, table_name: str, schema: StructType, mode: str = "append"):
    """
    Save DataFrame to Delta table in Databricks with an explicit schema.
    
    Args:
        df: Pandas DataFrame to save
        table_name: Name of the Delta table (e.g., 'garmin.activities')
        schema: Explicit schema for the target table
        mode: Write mode ('append', 'overwrite', etc.)
    """
    print(f"📊 Attempting to save DataFrame to {table_name} with mode='{mode}'")
    
    if df.empty:
        print(f"❌ No data to save to {table_name} - DataFrame is empty")
        return False
    
    try:
        # Convert pandas DataFrame to Spark DataFrame using the explicit schema
        print(f"🔄 Converting pandas DataFrame to Spark DataFrame with explicit schema...")
        spark_df = spark.createDataFrame(df, schema=schema)
        
        print(f"🔄 Writing to Delta table {table_name}...")
        # Write to Delta table
        spark_df.write \
            .format("delta") \
            .mode(mode) \
            .option("mergeSchema", "true") \
            .saveAsTable(table_name)
        
        print(f"✅ Successfully saved {len(df)} records to {table_name}")
        return True
        
    except Exception as e:
        print(f"❌ Error saving to {table_name}: {str(e)}")
        print(f"📊 DataFrame sample:")
        print(df.head())
        return False


# COMMAND ----------

# MAGIC %md
# MAGIC ## Main Execution Functions

# COMMAND ----------

def extract_garmin_activities(days_back: int = 30, limit: int = 100) -> bool:
    """
    Extract activities from Garmin Connect and save to Delta table
    
    Args:
        days_back: Number of days back to retrieve data
        limit: Maximum number of activities to retrieve
    """
    try:
        # Initialize API client
        garmin_api = GarminConnectAPI()
        
        # Authenticate
        if not garmin_api.authenticate():
            raise Exception("Failed to authenticate with Garmin Connect")
        
        # Calculate start date
        start_date = (datetime.now() - timedelta(days=days_back)).strftime("%Y-%m-%d")
        
        # Get activities
        print(f"Retrieving activities from {start_date}...")
        activities = garmin_api.get_activities(start_date=start_date, limit=limit)
        
        if not activities:
            print("No activities found")
            return True
        
        print(f"Found {len(activities)} activities")
        
        # Process to DataFrame
        df = process_activities_to_dataframe(activities)
        
        # Save to Delta table with explicit schema
        save_to_delta_table(df, "garmin.activities", schema=get_activities_schema(), mode="append")
        
        return True
        
    except Exception as e:
        print(f"Error in extract_garmin_activities: {str(e)}")
        return False

def extract_daily_summaries(days_back: int = 7) -> bool:
    """
    Extract daily summary data from Garmin Connect
    """
    try:
        garmin_api = GarminConnectAPI()
        
        if not garmin_api.authenticate():
            raise Exception("Failed to authenticate with Garmin Connect")
        
        daily_summaries = []
        
        # Get data for each day
        for i in range(days_back):
            date = (datetime.now() - timedelta(days=i)).strftime("%Y-%m-%d")
            print(f"Retrieving daily summary for {date}...")
            
            summary = garmin_api.get_daily_summary(date)
            
            if summary:
                summary['date'] = date
                summary['retrieved_at'] = datetime.now()
                daily_summaries.append(summary)
                
        print(f"📊 Total summaries collected: {len(daily_summaries)}")
        
        if daily_summaries:
            print(f"📊 Processing summaries...")
            df = process_daily_summary_to_dataframe(daily_summaries)
            
            if not df.empty:
                success = save_to_delta_table(df, "garmin.daily_summaries", schema=get_daily_summaries_schema(), mode="append")
                return success
            else:
                print("❌ Processed DataFrame is empty")
                return False
        else:
            print("❌ No daily summaries to save")
            return True
        
    except Exception as e:
        print(f"❌ Error in extract_daily_summaries: {str(e)}")
        return False

# COMMAND ----------

# MAGIC %md
# MAGIC ## Main
# MAGIC 
# MAGIC Run this as a scheduled job in Databricks.

# COMMAND ----------

# Run the data extraction
if __name__ == "__main__":
    print("Starting Garmin data extraction...")
    
    # Create database first (tables will be created on write)
    print("🔧 Setting up database...")
    if not create_garmin_database_and_tables():
        print("❌ Failed to create database, stopping extraction")
    else:
        # Extract activities from last 30 days
        success_activities = extract_garmin_activities(days_back=30, limit=200)
        
        # Extract daily summaries from last 7 days
        success_summaries = extract_daily_summaries(days_back=7)
        
        if success_activities and success_summaries:
            print("✅ Garmin data extraction completed successfully!")
        else:
            print("❌ Some errors occurred during extraction")
        
        # Show what we created
        print("\n📊 Checking created tables:")
        try:
            spark.sql("SHOW TABLES IN garmin").show()
        except Exception as e:
            print(f"Error showing tables: {e}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## Data Quality Checks

# COMMAND ----------

def run_data_quality_checks():
    """
    Run basic data quality checks on the extracted data
    """
    try:
        # Check activities table
        activities_df = spark.table("garmin.activities")
        activities_count = activities_df.count()
        print(f"Total activities in table: {activities_count}")
        
        # Check for recent data
        recent_activities = activities_df.filter(
            col("retrieved_at") >= date_sub(current_date(), 1)
        ).count()
        print(f"Activities retrieved in last 24 hours: {recent_activities}")
        
        # Check daily summaries
        summaries_df = spark.table("garmin.daily_summaries")
        summaries_count = summaries_df.count()
        print(f"Total daily summaries: {summaries_count}")
        
    except Exception as e:
        print(f"Error running data quality checks: {str(e)}")

# Uncomment to run data quality checks
run_data_quality_checks()

# COMMAND ----------

# MAGIC %md
# MAGIC ## Test Authentication

# COMMAND ----------

def test_authentication():
    """
    Test Garmin Connect authentication without retrieving data
    """
    try:
        garmin_api = GarminConnectAPI()
        
        if garmin_api.authenticate():
            print("✅ Authentication successful!")
            print("You can now run the full data extraction.")
            return True
        else:
            print("❌ Authentication failed!")
            print("Please check your credentials in the secret scope.")
            return False
            
    except Exception as e:
        print(f"❌ Authentication error: {str(e)}")
        return False


# Uncomment to test authentication
test_authentication()

In [0]:
%sql
select * from garmin.activities limit 10



In [0]:
%sql
select * from garmin.activity_details limit 10

In [0]:
%sql
select * from garmin.daily_summaries limit 10

In [0]:
# Uncomment to test authentication
test_authentication()

